In [1]:
import pandas as pd
import math
from collections import Counter

In [ ]:
data=pd.read_csv("play_tennis.csv")
dataset=data.values.tolist()
features=list(data.columns[:-1])

In [11]:
data.head()

,Outlook,Temperature,Humidity,Wind,Play
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


In [3]:
def entropy(data):
    labels=[row[-1] for row in data]
    counts=Counter(labels)
    total=len(labels)
    ent=0
    for count in counts.values():
        probability=count/total
        ent-=probability*math.log2(probability)
    return ent

In [4]:
def split_data(data,feature_index,value):
    subset=[]
    for row in data:
        if row[feature_index]==value:
            reduced_row=row[:feature_index]+row[feature_index+1:]
            subset.append(reduced_row)
    return subset

In [5]:
def information_gain(data, feature_index):
    total_entropy=entropy(data)
    values=set(row[feature_index] for row in data)
    weighted_entropy=0
    for value in values:
        subset=[row for row in data if row[feature_index]==value]
        probability= len(subset)/len(data)
        weighted_entropy+=probability*entropy(subset)
    gain=total_entropy-weighted_entropy
    return gain

In [6]:
def best_feature(data):
    num_features=len(data[0])-1
    gains=[]
    for i in range(num_features):
        gains.append(information_gain(data,i))
    return gains.index(max(gains))

def majority_class(data):
    labels=[row[-1] for row in data]
    return Counter(labels).most_common(1)[0][0]

def id3(data,features):
    labels=[row[-1]for row in data]
    if labels.count(labels[0])==len(labels):
        return labels[0]
    
    if len(features)==0:
        return majority_class(data)
    
    best=best_feature(data)
    best_feature_name=features[best]

    tree={best_feature_name: {}}

    values=set(row[best] for row in data)

    for value in values:
        subset=split_data(data,best,value)
        if len(subset)==0:
            tree[best_feature_name][value]=majority_class(data)
        else:
            remaining_features=features[:best]+features[best+1:]
        
        tree[best_feature_name][value]=id3(
            subset,
            remaining_features
        )
    return tree

decision_tree=id3(dataset,features)

print("\nDecision Tree\n")
print(decision_tree)


Decision Tree

{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}
